# CDIME-AI on Colab (Free T4)
**Causal Domain-Incremental Multimodal Explainable AI for Chest X-ray Diagnosis**

This notebook runs the full CDIME-AI framework end-to-end on a free Colab T4:
continual learning across CheXpert → MIMIC-CXR → VinDr-CXR, causal (IRM + V-REx)
regularisation, EWC + replay + domain adapters, Grad-CAM / attention /
counterfactual explanations, MC-Dropout + temperature-scaling uncertainty,
therapeutic decision support, ablations and statistical significance tests.

> **Runtime → Change runtime type → T4 GPU** before running.

It works **with no dataset downloads** using a causal synthetic generator (so
you get a complete results table in minutes), and includes adapters to plug in
the real datasets when you have them.

## 1. Setup

In [ ]:
# Clone the implementation (or upload the cdime_ai/ folder to /content)
import os
if not os.path.exists('/content/cdime_ai'):
    !git clone -b claude/q1-paper-implementation-kuzdmc https://github.com/HaiderShah786/machine-learning-python.git /content/mlp || true
    if os.path.exists('/content/mlp/cdime_ai'):
        !cp -r /content/mlp/cdime_ai /content/cdime_ai
import sys; sys.path.insert(0, '/content')
print('cdime_ai present:', os.path.exists('/content/cdime_ai'))

In [ ]:
!pip install -q timm transformers scikit-learn scipy >/dev/null 2>&1
import torch; print('CUDA available:', torch.cuda.is_available(),
                     '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configure (T4-friendly defaults)
`Config()` is already tuned for a 16 GB T4. Scale `train_per_domain` and
`epochs_per_domain` up for full Q1 results when you have more compute/time.

In [ ]:
from cdime_ai.config import Config
from cdime_ai.tokenizer import get_tokenizer
from cdime_ai.data.loaders import build_domain_loaders

cfg = Config()
cfg.data.mode = 'synthetic'         # 'real' to use on-disk CheXpert/MIMIC/VinDr
# T4 memory knobs (raise for full runs):
cfg.train.epochs_per_domain = 4
cfg.train.batch_size = 8
cfg.data.train_per_domain = 600
print('device:', cfg.device, '| domains:', cfg.domains, '| labels:', cfg.labels)

tokenizer = get_tokenizer(cfg)          # ClinicalBERT (auto-fallback to DistilBERT)
loaders = build_domain_loaders(cfg, tokenizer)

## 3. Run the continual protocol (Phase 1→2→3)

In [ ]:
from cdime_ai.engine.continual_runner import run_continual
results = run_continual(cfg, loaders, tokenizer)
model = results.pop('model')
print('\nContinual-learning metrics:')
for k, v in results['continual'].items():
    print(f'  {k:20s}: {v:.4f}')

In [ ]:
import numpy as np, pandas as pd
R = np.array(results['R_matrix'])
df = pd.DataFrame(R, index=[f'after {d}' for d in cfg.domains], columns=cfg.domains)
print('Accuracy matrix R[i,j] = acc on domain j after training through i:')
df.round(3)

## 4. Bootstrap confidence intervals (AUROC / F1)

In [ ]:
from cdime_ai.evaluation.metrics import collect_predictions
from cdime_ai.evaluation.stats import bootstrap_ci
last = len(cfg.domains)-1
probs, targets = collect_predictions(model, loaders[cfg.domains[last]]['test'], domain=last, device=cfg.device)
print('AUROC:', bootstrap_ci(probs, targets, 'auroc', n_boot=1000, seed=cfg.seed))
print('F1   :', bootstrap_ci(probs, targets, 'f1',    n_boot=1000, seed=cfg.seed))

## 5. Explainability — Grad-CAM, attention evidence, counterfactual

In [ ]:
import matplotlib.pyplot as plt, torch
from cdime_ai.utils import move_batch
from cdime_ai.explain.gradcam import GradCAM
from cdime_ai.explain.attention import image_to_text_attention, top_text_evidence
from cdime_ai.explain.counterfactual import generate_counterfactual
from cdime_ai.uncertainty.mc_dropout import mc_dropout_predict

batch = move_batch(next(iter(loaders[cfg.domains[last]]['test'])), cfg.device)
mean_p, std_p, ent = mc_dropout_predict(model, batch, last, n_samples=20)
cls = int(mean_p[0].argmax())
cam = GradCAM(model)(batch, class_idx=cls, domain=last)

img = batch['image'][0].cpu()
img = (img - img.min())/(img.max()-img.min())
fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(img.permute(1,2,0)); ax[0].set_title('Input CXR'); ax[0].axis('off')
ax[1].imshow(cam[0].cpu(), cmap='jet'); ax[1].set_title(f'Grad-CAM: {cfg.labels[cls]}'); ax[1].axis('off')
ax[2].imshow(img.permute(1,2,0)); ax[2].imshow(cam[0].cpu(), cmap='jet', alpha=0.5); ax[2].set_title('Overlay'); ax[2].axis('off')
plt.tight_layout(); plt.show()

attn = image_to_text_attention(model, batch, domain=last)
print('Top report evidence:', top_text_evidence(attn[0] if attn is not None else None, batch['input_ids'][0], tokenizer, k=5))
cf, delta, p0, p1 = generate_counterfactual(model, batch, class_idx=cls, domain=last)
print(f'Counterfactual {cfg.labels[cls]}: p {p0:.2f} -> {p1:.2f}')

## 6. Decision support report

In [ ]:
from cdime_ai.decision.therapeutic import make_decision
report = make_decision(cfg.labels, mean_p[0].cpu().numpy(), std_p[0].cpu().numpy(),
                       evidence=[f'{t} ({w:.2f})' for t,w in top_text_evidence(
                           attn[0] if attn is not None else None, batch['input_ids'][0], tokenizer, 3)])
print(report.to_text())

## 7. Ablation study + significance tests (optional, longer)
Runs all 7 ablations + baselines over multiple seeds and runs paired
t-test/Wilcoxon vs. the full model. Reduce seeds / sizes for a quick look.

In [ ]:
from cdime_ai.ablation import run_ablation_study
from cdime_ai.main import _print_ablation_table
# A quick 2-seed pass; use [42,43,44,45,46] for the paper.
study = run_ablation_study(cfg, tokenizer, build_domain_loaders, seeds=[42,43])
_print_ablation_table(study['summary'])

## 8. Using the REAL datasets
1. Download subsets and arrange CSVs as described in
   `cdime_ai/data/datasets.py` (`REAL_PATHS`).
2. Set `cfg.data.mode = 'real'` and rebuild loaders.

The rest of the pipeline is unchanged — every component is dataset-agnostic.
On a free T4, use subsets (a few thousand studies per domain); the full
CheXpert/MIMIC-CXR cannot be downloaded on Colab free tier.